In [ ]:
# Example 1: URL canonicalization and safe-URL handling.
from w3lib.url import canonicalize_url, safe_url_string

url = "http://www.example.com/do?a=1&b=2&a=2&c=3"
canonical = canonicalize_url(url)
print(f"canonical: {canonical}")

unsafe = "http://www.example.com/path with spaces/é"
safe = safe_url_string(unsafe)
print(f"safe: {safe}")

assert canonical.startswith("http://www.example.com/do?")
assert "a=1" in canonical and "a=2" in canonical
assert "%20" in safe or "+" in safe


In [ ]:
# Example 2: URL query manipulation and cleaning.
# (urljoin_rfc was removed in w3lib 2.0 — url_query_cleaner is the current API.)
from w3lib.url import add_or_replace_parameter, url_query_cleaner, url_query_parameter

base = "http://example.com/search?q=cats&page=1"
new_url = add_or_replace_parameter(base, "page", "2")
print(f"new_url: {new_url}")

page_val = url_query_parameter(new_url, "page")
print(f"page param: {page_val}")

cleaned = url_query_cleaner(new_url, ["q"])
print(f"cleaned: {cleaned!r}")

assert page_val == "2"
assert "page=2" in new_url
assert cleaned == "http://example.com/search?q=cats"

In [ ]:
# Example 3: HTML utilities -- replace entities, strip tags, get base URL.
from w3lib.html import (
    get_base_url,
    remove_tags,
    replace_entities,
    replace_escape_chars,
)

html = b"<html><head><base href='http://example.com/'></head><body><p>Hello&amp;world</p></body></html>"
base = get_base_url(html, baseurl="http://fallback.com/")
print(f"base: {base}")

stripped = remove_tags("<p>Hello <b>world</b></p>")
print(f"stripped: {stripped}")

decoded = replace_entities("AT&amp;T &lt;3")
print(f"decoded: {decoded}")

no_esc = replace_escape_chars("a\tb\nc")
print(f"no_esc: {no_esc!r}")

assert base == "http://example.com/"
assert stripped == "Hello world"
assert decoded == "AT&T <3"


In [ ]:
# Example 4: HTTP basic-auth header construction.
from w3lib.http import basic_auth_header, headers_raw_to_dict, headers_dict_to_raw

header = basic_auth_header("user", "hunter2")
print(f"basic_auth: {header!r}")

raw = b"Content-Type: text/html\r\nContent-Length: 42\r\n"
as_dict = headers_raw_to_dict(raw)
print(f"as_dict: {as_dict}")

round_trip = headers_dict_to_raw(as_dict)
print(f"round_trip: {round_trip!r}")

assert header.startswith(b"Basic ")
assert b"Content-Type" in as_dict
assert b"Content-Type" in round_trip


In [ ]:
# Example 5: Encoding detection from HTTP headers + HTML body.
from w3lib.encoding import (
    html_body_declared_encoding,
    http_content_type_encoding,
    resolve_encoding,
    html_to_unicode,
)

http_enc = http_content_type_encoding("text/html; charset=utf-8")
print(f"http_enc: {http_enc}")

html = b"<html><head><meta charset='iso-8859-1'></head><body>x</body></html>"
body_enc = html_body_declared_encoding(html)
print(f"body_enc: {body_enc}")

resolved = resolve_encoding("latin1")
print(f"resolved: {resolved}")

content_type, text = html_to_unicode("text/html; charset=utf-8", b"<p>hi</p>")
print(f"content_type={content_type} text={text!r}")

assert http_enc == "utf-8"
assert body_enc in ("iso-8859-1", "cp1252")
assert resolved == "cp1252"
assert text == "<p>hi</p>"
